In [ ]:
import socket

# 1. Wide RDD across many partitions
rdd = sc.parallelize(range(1, 2_000_001), numSlices=200)
squares = rdd.filter(lambda x: x % 2 == 0).map(lambda x: x * x)
print("Even squares count:", squares.count())

In [ ]:
# 2. Prove which physical hosts actually processed partitions
def partition_host(idx, iterator):
    host = socket.gethostname()
    n = sum(1 for _ in iterator)
    yield (host, idx, n)

In [ ]:
for host, idx, n in sorted(rdd.mapPartitionsWithIndex(partition_host).collect()):
    print(f"partition {idx:3d} -> {host:20s} ({n} records)")

In [ ]:
# 3. Shuffle-heavy: reduceByKey (partial combine before shuffle)
pairs = sc.parallelize([(i % 50, 1) for i in range(1_000_000)], numSlices=100)
counts = dict(pairs.reduceByKey(lambda a, b: a + b).collect())
print("Key 0 count:", counts[0])

In [ ]:
# 4. groupByKey — heavier shuffle (ships every raw value, no pre-combine)
grouped = pairs.groupByKey().mapValues(len).collect()
print("Sample grouped:", sorted(grouped)[:5])

In [ ]:
from pyspark.sql import functions as F
import socket

In [ ]:
df1 = spark.range(0, 2_000_000).withColumn("key", F.col("id") % 1000).repartition(64)
df2 = spark.range(0, 1000).withColumnRenamed("id", "key") \
           .withColumn("label", F.concat(F.lit("grp_"), F.col("key")))

In [ ]:
# Wide aggregation -> triggers an Exchange (shuffle) stage
agg = df1.groupBy("key").agg(F.count("*").alias("cnt"), F.avg("id").alias("avg_id"))
agg.orderBy(F.desc("cnt")).show(5)

In [ ]:
# Shuffle join across the two DataFrames
joined = df1.join(df2, on="key", how="inner")
print("Joined row count:", joined.count())

In [ ]:
def host_of_partition(idx, iterator):
    yield (socket.gethostname(), idx, sum(1 for _ in iterator))

In [ ]:
for h, idx, n in sorted(joined.rdd.mapPartitionsWithIndex(host_of_partition).collect()):
    print(f"partition {idx:3d} -> {h:20s} ({n} rows)")